In [0]:
# Trust Maintenance Incremental - corrected implementation
#
# This notebook consumes durable encounter-remapping events produced by the
# updated classification notebook. It never refreshes the trust map from a
# second hard-coded list, never overwrites ADC_UPDT, resolves NULL trust before
# BHRUT cleanup, archives exclusions, records table-level metrics, and fails the
# task when any table operation fails.

In [0]:
import json
import time
import uuid
from datetime import datetime, timezone
from functools import reduce

from delta.tables import DeltaTable
from pyspark.sql import Window
from pyspark.sql import functions as F

In [0]:
def _widget(name, default, choices=None):
    try:
        if choices:
            dbutils.widgets.dropdown(name, default, choices)
        else:
            dbutils.widgets.text(name, default)
        return dbutils.widgets.get(name)
    except Exception:
        return default


TARGET_ENV = _widget("target_env", "prod", ["prod", "dev"]).lower()
DRY_RUN = _widget("dry_run", "false", ["false", "true"]).lower() == "true"
TABLE_FILTER = _widget("table_filter", "").strip().lower()
OPTIMIZE_LOOKUPS = _widget("optimize_lookups", "false", ["false", "true"]).lower() == "true"

CATALOG = "4_prod" if TARGET_ENV == "prod" else "8_dev"
RAW_SCHEMA = "raw"
TMP_SCHEMA = "tmp"
RAW_EXCLUDED_SCHEMA = "raw_excluded"
STAGING_SCHEMA = "staging"
LOG_CATALOG = "6_mgmt"
LOG_SCHEMA = "logs"

RUN_ID = str(uuid.uuid4())
RUN_STARTED_AT = datetime.now(timezone.utc)

TRUST_MAP_TBL = f"{CATALOG}.{TMP_SCHEMA}.org_to_trust_map"
POLICY_TBL = f"{CATALOG}.{TMP_SCHEMA}.trust_table_policy"
ENC_ORG_TBL = f"{CATALOG}.{TMP_SCHEMA}.sw_enc_org_map"
HUB_TBL = f"{CATALOG}.{TMP_SCHEMA}.sw_mapping_hub"
CHANGED_ENC_TBL = f"{CATALOG}.{TMP_SCHEMA}.sw_changed_encounters"
RUN_LOG_TBL = f"{LOG_CATALOG}.{LOG_SCHEMA}.trust_maintenance_run_log_v2"
TABLE_LOG_TBL = f"{LOG_CATALOG}.{LOG_SCHEMA}.trust_maintenance_table_log_v2"

spark.sql(f"USE CATALOG {CATALOG}")

print("=" * 88)
print("TRUST MAINTENANCE INCREMENTAL - UPDATED")
print(f"run_id={RUN_ID} env={TARGET_ENV} dry_run={DRY_RUN}")
print("=" * 88)

In [0]:
def table_exists(fqn):
    return spark.catalog.tableExists(fqn)


def table_columns(fqn):
    return spark.table(fqn).columns


def upper_map(columns):
    return {c.upper(): c for c in columns}


def latest_metrics(fqn, operation=None):
    for row in spark.sql(f"DESCRIBE HISTORY {fqn} LIMIT 20").collect():
        if operation is None or row["operation"] == operation:
            return dict(row["operationMetrics"] or {})
    return {}


def row_fingerprint(columns, alias=None):
    prefix = f"{alias}." if alias else ""
    values = [
        F.coalesce(F.col(f"{prefix}{c}").cast("string"), F.lit("<NULL>")).alias(c)
        for c in columns
    ]
    return F.sha2(F.to_json(F.struct(*values)), 256)


def write_run_log(status, table_count, failed_count, notes=None):
    values = [(
        RUN_ID, RUN_STARTED_AT, datetime.now(timezone.utc), TARGET_ENV,
        status, int(table_count), int(failed_count), notes,
    )]
    schema = (
        "run_id string, started_at timestamp, ended_at timestamp, target_env string, "
        "status string, table_count long, failed_count long, notes string"
    )
    spark.createDataFrame(values, schema).write.mode("append").saveAsTable(RUN_LOG_TBL)


def write_table_log(table_name, action, status, rows_affected=0, error_message=None):
    values = [(
        RUN_ID, datetime.now(timezone.utc), table_name, action, status,
        int(rows_affected or 0), error_message,
    )]
    schema = (
        "run_id string, logged_at timestamp, table_name string, action string, "
        "status string, rows_affected long, error_message string"
    )
    spark.createDataFrame(values, schema).write.mode("append").saveAsTable(TABLE_LOG_TBL)


def ensure_setup():
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {LOG_CATALOG}.{LOG_SCHEMA}")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{RAW_EXCLUDED_SCHEMA}")
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {RUN_LOG_TBL} (
            run_id STRING,
            started_at TIMESTAMP,
            ended_at TIMESTAMP,
            target_env STRING,
            status STRING,
            table_count BIGINT,
            failed_count BIGINT,
            notes STRING
        ) USING DELTA
    """)
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {TABLE_LOG_TBL} (
            run_id STRING,
            logged_at TIMESTAMP,
            table_name STRING,
            action STRING,
            status STRING,
            rows_affected BIGINT,
            error_message STRING
        ) USING DELTA
    """)
    required = [TRUST_MAP_TBL, POLICY_TBL, ENC_ORG_TBL, CHANGED_ENC_TBL]
    missing = [fqn for fqn in required if not table_exists(fqn)]
    if missing:
        raise RuntimeError(
            "Missing shared trust infrastructure. Run the updated classification "
            f"notebook setup first. Missing: {missing}"
        )
    changed_cols = upper_map(table_columns(CHANGED_ENC_TBL))
    required_changed = {"NEW_ORGANIZATION_ID", "RUN_ID", "PROCESSED_AT"}
    if not required_changed.issubset(changed_cols):
        raise RuntimeError(
            f"{CHANGED_ENC_TBL} does not have the durable-v2 columns. "
            "Run the updated classification notebook setup first."
        )

    # Normalize historical casing without replacing the table or its identity.
    if not DRY_RUN:
        spark.sql(f"""
            UPDATE {TRUST_MAP_TBL}
            SET trust = CASE
                WHEN UPPER(trust) = 'BHRUT' THEN 'BHRUT'
                WHEN UPPER(trust) = 'BARTS' THEN 'Barts'
                ELSE trust
            END
            WHERE trust IS NOT NULL
        """)


def discover_raw_tables():
    requested = {x.strip() for x in TABLE_FILTER.split(",") if x.strip()}
    names = []
    for row in spark.sql(f"SHOW TABLES IN {CATALOG}.{RAW_SCHEMA}").collect():
        name = row["tableName"]
        if not name.lower().startswith("mill_"):
            continue
        if requested and name.lower() not in requested:
            continue
        try:
            if "TRUST" in upper_map(table_columns(f"{CATALOG}.{RAW_SCHEMA}.{name}")):
                names.append(name)
        except Exception:
            continue
    return sorted(names)


def policy_map():
    return {
        row["table_name"].lower(): bool(row["keep_bhrut"])
        for row in spark.table(POLICY_TBL).collect()
    }


def pending_encounter_changes():
    pending = spark.table(CHANGED_ENC_TBL).filter(F.col("processed_at").isNull())
    if pending.limit(1).count() == 0:
        return pending

    window = Window.partitionBy("ENCNTR_ID").orderBy(
        F.col("detected_at").desc_nulls_last(),
        F.col("run_id").desc_nulls_last(),
    )
    latest = pending.withColumn("_rn", F.row_number().over(window)).filter(
        F.col("_rn") == 1
    )
    current_enc = spark.table(ENC_ORG_TBL).select(
        "ENCNTR_ID", F.col("ORGANIZATION_ID").alias("_current_organization_id")
    )
    trust_map = spark.table(TRUST_MAP_TBL).select(
        F.col("organization_id").alias("_trust_organization_id"),
        F.col("trust").alias("_new_trust"),
    )
    return (
        latest.join(current_enc, "ENCNTR_ID", "left")
        .withColumn(
            "_effective_organization_id",
            F.coalesce(F.col("new_organization_id"), F.col("_current_organization_id")),
        )
        .join(
            F.broadcast(trust_map),
            F.col("_effective_organization_id") == F.col("_trust_organization_id"),
            "left",
        )
        .select(
            F.col("ENCNTR_ID").cast("long").alias("ENCNTR_ID"),
            F.col("_new_trust").alias("Trust"),
        )
        .dropDuplicates(["ENCNTR_ID"])
    )


def propagate_remaps(raw_tables):
    pending = pending_encounter_changes()
    pending_count = pending.count()
    if pending_count == 0:
        print("No unprocessed encounter remaps.")
        return []

    print(f"Propagating {pending_count:,} encounter remaps.")
    errors = []
    for table_name in raw_tables:
        fqn = f"{CATALOG}.{RAW_SCHEMA}.{table_name}"
        cols = upper_map(table_columns(fqn))
        if "ENCNTR_ID" not in cols:
            continue
        try:
            source = pending.select(
                F.col("ENCNTR_ID").alias(cols["ENCNTR_ID"]),
                F.col("Trust").alias(cols["TRUST"]),
            )
            if DRY_RUN:
                affected = (
                    spark.table(fqn).alias("t")
                    .join(source.alias("s"), cols["ENCNTR_ID"])
                    .filter(~F.col(f"t.{cols['TRUST']}").eqNullSafe(F.col(f"s.{cols['TRUST']}")))
                    .count()
                )
            else:
                DeltaTable.forName(spark, fqn).alias("t").merge(
                    source.alias("s"),
                    F.col(f"t.{cols['ENCNTR_ID']}") == F.col(f"s.{cols['ENCNTR_ID']}"),
                ).whenMatchedUpdate(
                    condition=~F.col(f"t.{cols['TRUST']}").eqNullSafe(F.col(f"s.{cols['TRUST']}")),
                    set={cols["TRUST"]: F.col(f"s.{cols['TRUST']}")},
                ).execute()
                metrics = latest_metrics(fqn, "MERGE")
                affected = int(metrics.get("numTargetRowsUpdated", 0))
            write_table_log(table_name, "PROPAGATE_REMAP", "DRY_RUN" if DRY_RUN else "SUCCESS", affected)
        except Exception as exc:
            message = f"{type(exc).__name__}: {exc}"
            errors.append((table_name, message))
            write_table_log(table_name, "PROPAGATE_REMAP", "FAILED", 0, message[:8000])

    if not errors and not DRY_RUN:
        DeltaTable.forName(spark, CHANGED_ENC_TBL).update(
            condition=F.col("processed_at").isNull(),
            set={"processed_at": F.current_timestamp()},
        )
    return errors


def resolve_null_trust(raw_tables):
    errors = []
    trust_map = spark.table(TRUST_MAP_TBL).select(
        F.col("organization_id").alias("_organization_id"),
        F.col("trust").alias("_trust"),
    )
    enc_trust = (
        spark.table(ENC_ORG_TBL).alias("eo")
        .join(
            F.broadcast(trust_map.alias("tm")),
            F.col("eo.ORGANIZATION_ID") == F.col("tm._organization_id"),
            "left",
        )
        .select(
            F.col("eo.ENCNTR_ID").cast("long").alias("_encntr_id"),
            F.col("tm._trust").alias("_trust"),
        )
        .dropDuplicates(["_encntr_id"])
    )

    for table_name in raw_tables:
        fqn = f"{CATALOG}.{RAW_SCHEMA}.{table_name}"
        cols = upper_map(table_columns(fqn))
        try:
            total_updated = 0
            if "ORGANIZATION_ID" in cols:
                source = trust_map.filter(F.col("_trust").isNotNull())
                if DRY_RUN:
                    affected = (
                        spark.table(fqn).alias("t")
                        .join(
                            source.alias("s"),
                            F.col(f"t.{cols['ORGANIZATION_ID']}") == F.col("s._organization_id"),
                        )
                        .filter(F.col(f"t.{cols['TRUST']}").isNull())
                        .count()
                    )
                else:
                    DeltaTable.forName(spark, fqn).alias("t").merge(
                        source.alias("s"),
                        F.col(f"t.{cols['ORGANIZATION_ID']}") == F.col("s._organization_id"),
                    ).whenMatchedUpdate(
                        condition=F.col(f"t.{cols['TRUST']}").isNull(),
                        set={cols["TRUST"]: F.col("s._trust")},
                    ).execute()
                    affected = int(latest_metrics(fqn, "MERGE").get("numTargetRowsUpdated", 0))
                total_updated += affected

            if "ENCNTR_ID" in cols:
                source = enc_trust.filter(F.col("_trust").isNotNull())
                if DRY_RUN:
                    affected = (
                        spark.table(fqn).alias("t")
                        .join(
                            source.alias("s"),
                            F.col(f"t.{cols['ENCNTR_ID']}") == F.col("s._encntr_id"),
                        )
                        .filter(F.col(f"t.{cols['TRUST']}").isNull())
                        .count()
                    )
                else:
                    DeltaTable.forName(spark, fqn).alias("t").merge(
                        source.alias("s"),
                        F.col(f"t.{cols['ENCNTR_ID']}") == F.col("s._encntr_id"),
                    ).whenMatchedUpdate(
                        condition=F.col(f"t.{cols['TRUST']}").isNull(),
                        set={cols["TRUST"]: F.col("s._trust")},
                    ).execute()
                    affected = int(latest_metrics(fqn, "MERGE").get("numTargetRowsUpdated", 0))
                total_updated += affected

            write_table_log(
                table_name, "RESOLVE_NULL_TRUST",
                "DRY_RUN" if DRY_RUN else "SUCCESS", total_updated,
            )
        except Exception as exc:
            message = f"{type(exc).__name__}: {exc}"
            errors.append((table_name, message))
            write_table_log(table_name, "RESOLVE_NULL_TRUST", "FAILED", 0, message[:8000])
    return errors


def ensure_archive_table(target_fqn, source_df):
    if not table_exists(target_fqn):
        source_df.limit(0).write.format("delta").saveAsTable(target_fqn)
        return
    target_upper = upper_map(table_columns(target_fqn))
    for field in source_df.schema.fields:
        if field.name.upper() not in target_upper:
            spark.sql(
                f"ALTER TABLE {target_fqn} ADD COLUMNS "
                f"({field.name} {field.dataType.simpleString()})"
            )


def archive_raw_exclusions(table_name, rows_df):
    if rows_df.limit(1).count() == 0:
        return 0
    source_columns = rows_df.columns
    archived = (
        rows_df.withColumn("_row_fingerprint", row_fingerprint(source_columns))
        .withColumn("_archive_reason", F.lit("BHRUT_FILTERED"))
        .withColumn("_archived_at", F.current_timestamp())
        .withColumn("_run_id", F.lit(RUN_ID))
        .dropDuplicates(["_row_fingerprint"])
    )
    count = archived.count()
    if DRY_RUN:
        return count

    target_fqn = f"{CATALOG}.{RAW_EXCLUDED_SCHEMA}.{table_name}"
    ensure_archive_table(target_fqn, archived)
    target_upper = upper_map(table_columns(target_fqn))
    source_upper = upper_map(archived.columns)
    values = {
        target_upper[u]: F.col(f"s.{source_upper[u]}")
        for u in target_upper
        if u in source_upper
    }
    DeltaTable.forName(spark, target_fqn).alias("t").merge(
        archived.alias("s"),
        F.col("t._row_fingerprint") == F.col("s._row_fingerprint"),
    ).whenNotMatchedInsert(values=values).execute()
    return count


def cleanup_bhrut(raw_tables, policies):
    errors = []
    for table_name in raw_tables:
        if policies.get(table_name.lower(), False):
            write_table_log(table_name, "CLEANUP_BHRUT", "PROTECTED", 0)
            continue
        fqn = f"{CATALOG}.{RAW_SCHEMA}.{table_name}"
        cols = upper_map(table_columns(fqn))
        try:
            bhrut = spark.table(fqn).filter(F.upper(F.col(cols["TRUST"])) == "BHRUT")
            count = bhrut.count()
            if count:
                archive_raw_exclusions(table_name, bhrut)
                if not DRY_RUN:
                    DeltaTable.forName(spark, fqn).delete(
                        F.upper(F.col(cols["TRUST"])) == "BHRUT"
                    )
            write_table_log(
                table_name, "CLEANUP_BHRUT",
                "DRY_RUN" if DRY_RUN else "SUCCESS", count,
            )
        except Exception as exc:
            message = f"{type(exc).__name__}: {exc}"
            errors.append((table_name, message))
            write_table_log(table_name, "CLEANUP_BHRUT", "FAILED", 0, message[:8000])
    return errors


def staging_health_report():
    rows = []
    for row in spark.sql(f"SHOW TABLES IN {CATALOG}.{STAGING_SCHEMA}").collect():
        table_name = row["tableName"]
        if not table_name.lower().startswith("mill_"):
            continue
        fqn = f"{CATALOG}.{STAGING_SCHEMA}.{table_name}"
        cols = upper_map(table_columns(fqn))
        if "_STAGED_AT" not in cols or "_CLASSIFICATION_ATTEMPTS" not in cols:
            continue
        stats = spark.table(fqn).agg(
            F.count("*").alias("rows"),
            F.min(F.col(cols["_STAGED_AT"])).alias("oldest"),
            F.max(F.col(cols["_CLASSIFICATION_ATTEMPTS"])).alias("max_attempts"),
        ).first()
        if stats["rows"]:
            rows.append((table_name, stats["rows"], stats["oldest"], stats["max_attempts"]))
    print(f"Staging tables with pending rows: {len(rows)}")
    for table_name, count, oldest, max_attempts in sorted(rows, key=lambda x: -x[1])[:30]:
        print(
            f"  {table_name:<42} rows={count:>12,} "
            f"oldest={str(oldest)[:19]} max_attempts={max_attempts}"
        )


def optimize_lookups():
    if not OPTIMIZE_LOOKUPS or DRY_RUN:
        return
    spark.sql(f"OPTIMIZE {ENC_ORG_TBL} ZORDER BY (ENCNTR_ID)")
    spark.sql(f"OPTIMIZE {HUB_TBL} ZORDER BY (key_type, key_id)")
    # VACUUM is intentionally omitted so recovery/time-travel is not shortened.


In [0]:
errors = []
raw_tables = []

try:
    ensure_setup()
    raw_tables = discover_raw_tables()
    policies = policy_map()

    errors.extend(propagate_remaps(raw_tables))
    errors.extend(resolve_null_trust(raw_tables))
    errors.extend(cleanup_bhrut(raw_tables, policies))
    staging_health_report()
    optimize_lookups()

    status = "FAILED" if errors else ("DRY_RUN" if DRY_RUN else "SUCCESS")
    write_run_log(
        status, len(raw_tables), len(errors),
        json.dumps(errors)[:16000] if errors else None,
    )
finally:
    print("=" * 88)
    print(f"run_id={RUN_ID} tables={len(raw_tables)} failures={len(errors)} dry_run={DRY_RUN}")
    print("=" * 88)

if errors:
    raise RuntimeError(
        f"Trust maintenance failed for {len(errors)} table action(s): "
        + "; ".join(f"{table}: {message}" for table, message in errors[:10])
    )

result = {
    "run_id": RUN_ID,
    "status": "DRY_RUN" if DRY_RUN else "SUCCESS",
    "tables": len(raw_tables),
    "failures": 0,
}
try:
    dbutils.notebook.exit(json.dumps(result))
except Exception:
    print(json.dumps(result))